# 07 — Visual review and annotation of garden images

Run from the **repository root** in VS Code/Jupyter. This notebook displays a mosaic, lets you select an image and correct its class, then exports a reviewed CSV and JSON. It **does not move files** or change the training folders automatically. All labels are provisional until reviewed.

In [ ]:
from pathlib import Path
import csv, json, hashlib
from datetime import datetime, timezone
from IPython.display import display, clear_output
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import ipywidgets as widgets

ROOT = Path.cwd()
if not (ROOT / "datasets" / "images").exists():
    raise FileNotFoundError("Open the notebook with the working directory set to the GardnAIze repository root")
DATA = ROOT / "datasets" / "images"
MANIFEST = DATA / "garden_image_provenance.csv"
REVIEW_CSV = DATA / "garden_image_reviews.csv"
REVIEW_JSON = DATA / "garden_image_reviews.json"
CLASSES = ("garden", "not_garden", "unusable", "exclude")
images = sorted(p for split in ("garden_validation", "garden_test") for cls in CLASSES[:3]
                for p in (DATA / split / cls).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"{len(images)} images found")
if not images:
    raise RuntimeError("No images found. Run the downloader first.")
provenance = {}
if MANIFEST.exists():
    with MANIFEST.open(newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            provenance[Path(row["file"]).name] = row
reviews = {}
if REVIEW_JSON.exists():
    existing = json.loads(REVIEW_JSON.read_text(encoding="utf-8"))
    reviews = {r["file"]: r for r in existing.get("reviews", [])}
print(f"{len(reviews)} existing reviews loaded")

## Mosaic

Choose the dataset split and class, then navigate by page. Use the selector beneath the mosaic to inspect and correct one image at a time.

In [ ]:
split_select = widgets.Dropdown(options=["all", "garden_validation", "garden_test"], description="Split")
class_select = widgets.Dropdown(options=["all", *CLASSES[:3]], description="Class")
page_select = widgets.BoundedIntText(value=1, min=1, max=9999, description="Page")
page_size = 20
mosaic_out = widgets.Output()

def filtered_images():
    return [p for p in images if (split_select.value == "all" or p.parent.parent.name == split_select.value)
            and (class_select.value == "all" or p.parent.name == class_select.value)]

def render_mosaic(*_):
    subset = filtered_images()
    pages = max(1, (len(subset) + page_size - 1) // page_size)
    page_select.max = pages
    page_select.value = min(page_select.value, pages)
    current = subset[(page_select.value - 1) * page_size:page_select.value * page_size]
    with mosaic_out:
        clear_output(wait=True)
        if not current:
            print("No images in this selection")
            return
        fig, axes = plt.subplots(4, 5, figsize=(16, 12))
        for ax, p in zip(axes.flat, current):
            try:
                with Image.open(p) as im:
                    ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
            except Exception as exc:
                ax.text(0.05, 0.5, f"Unreadable: {exc}", wrap=True)
            key = str(p.relative_to(ROOT))
            label = reviews.get(key, {}).get("corrected_label", p.parent.name)
            ax.set_title(f"{p.name[:21]}\n{label}", fontsize=8)
            ax.axis("off")
        for ax in list(axes.flat)[len(current):]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        print(f"Page {page_select.value}/{pages} | {len(subset)} images in selection")

for w in (split_select, class_select, page_select):
    w.observe(render_mosaic, names="value")
display(widgets.HBox([split_select, class_select, page_select]), mosaic_out)
render_mosaic()

## Correct annotations

Select a filename, choose its corrected class, optionally add a note and click **Save review**. A review record is written immediately to local CSV and JSON. No training image is moved, so existing notebook training labels remain folder-based until you apply the reviewed annotations.

In [ ]:
image_select = widgets.Dropdown(options=[(str(p.relative_to(DATA)), str(p.relative_to(ROOT))) for p in images],
                                description="Image", layout=widgets.Layout(width="800px"))
label_select = widgets.Dropdown(options=CLASSES, description="Correct label")
note_field = widgets.Text(description="Note", layout=widgets.Layout(width="800px"))
save_button = widgets.Button(description="Save review", button_style="success")
preview_out = widgets.Output()
status_out = widgets.Output()

def preview(*_):
    path = ROOT / image_select.value
    key = str(path.relative_to(ROOT))
    label_select.value = reviews.get(key, {}).get("corrected_label", path.parent.name)
    note_field.value = reviews.get(key, {}).get("note", "")
    with preview_out:
        clear_output(wait=True)
        with Image.open(path) as im:
            display(ImageOps.exif_transpose(im).convert("RGB").copy().resize((min(im.width, 700), int(im.height * min(im.width, 700) / im.width))))

def save_reviews():
    rows = [reviews[k] for k in sorted(reviews)]
    payload = {"schema_version": 1, "generated_at": datetime.now(timezone.utc).isoformat(),
               "note": "Human review; model predictions are not included", "reviews": rows}
    REVIEW_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    with REVIEW_CSV.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=["file", "original_label", "corrected_label", "note", "reviewed_at", "sha256", "commons_page", "license"])
        writer.writeheader()
        writer.writerows(rows)

def save_clicked(_):
    path = ROOT / image_select.value
    key = str(path.relative_to(ROOT))
    meta = provenance.get(path.name, {})
    reviews[key] = {
        "file": key, "original_label": path.parent.name, "corrected_label": label_select.value,
        "note": note_field.value, "reviewed_at": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "commons_page": meta.get("commons_page", ""), "license": meta.get("license", "")
    }
    save_reviews()
    with status_out:
        clear_output(wait=True)
        print(f"Saved {len(reviews)} reviews to {REVIEW_JSON.relative_to(ROOT)} and {REVIEW_CSV.relative_to(ROOT)}")
    render_mosaic()

image_select.observe(preview, names="value")
save_button.on_click(save_clicked)
display(image_select, label_select, note_field, save_button, preview_out, status_out)
preview()

## Review summary and next steps

Only reviewed labels in the JSON/CSV reflect human corrections. The existing training notebook reads class names from directories, **not** this review file. Before training, apply the corrections by moving reviewed files to the correct class folder **within the same split**, remove excluded files, and update provenance paths. Never move images between training and test splits without checking source/group leakage. The JSON here contains annotation reviews, **not ResNet-18 predictions**.

In [ ]:
from collections import Counter
print("Reviewed images:", len(reviews), "/", len(images))
print("Corrected labels:", dict(Counter(r["corrected_label"] for r in reviews.values())))
print("Changed labels:", sum(r["original_label"] != r["corrected_label"] for r in reviews.values()))
print("Reviews CSV:", REVIEW_CSV)
print("Reviews JSON:", REVIEW_JSON)